In [7]:
import argparse
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
import pandas as pd

# 读取数据
packets_df = pd.read_csv("dataset/firewall_test_20250731_160700-stage1_packets.csv")
flows_df = pd.read_csv("dataset/firewall_test_20250731_160700-stage1_flows.csv")

# 统计每个 flow 的包数（假设 packet_id 列存在且无缺失）
packet_counts = packets_df.groupby("flow_id")["packet_id"].count().reset_index()
packet_counts.columns = ["flow_id", "packet_count"]

# 选择 flows 中需要的列（flow_id 和 label）
flows_subset = flows_df[["flow_id", "label"]].copy()

# 合并两个 DataFrame：左连接保证所有 flows 都有记录（即使某个 flow 在 packets 中没有包，也会显示，但通常每个 flow 至少有一个包）
merged = packet_counts.merge(flows_subset, on="flow_id", how="outer")

# 如果某个 flow 在 packets 中没有包，会显示 NaNs，可以填充为 0（可选）
merged["packet_count"] = merged["packet_count"].fillna(0).astype(int)

merged.to_csv("flow_packet_count_with_label.csv", index=False)


In [8]:
flows = Path("dataset/firewall_test_20250731_160700-stage1_packets.csv")

if not flows.exists():
        raise FileNotFoundError(f"file not found: {flows}")

df = pd.read_csv(flows, low_memory=False)
# df.columns = [normalize_column_name(c) for c in df.columns]
# 去除掉flow_id为0的数据
df = df[df['flow_id'] != 0]


print(f"[OK] Loaded: {flows}")
print(f"     shape = {df.shape}")

[OK] Loaded: dataset\firewall_test_20250731_160700-stage1_packets.csv
     shape = (1003219, 40)


In [5]:
label0 = (df['label'] == 0).sum()
label1 = (df['label'] == 1).sum()
print("label0: ", label0)
print("label1: ", label1)

label0:  405255
label1:  16644


In [4]:
# 方法1：直接计算唯一值数量
unique_flow_count = df['flow_id'].nunique()
print(f"唯一的 flow_id 数量: {unique_flow_count}")

唯一的 flow_id 数量: 5199


In [6]:
# 按 flow_id 分组，计算每个 flow_id 的 packet_id 数量
packet_counts = df.groupby("flow_id")["packet_id"].count().reset_index()

# 重命名列以便理解
packet_counts.columns = ["flow_id", "packet_count"]

# 打印结果（前20行）
print(packet_counts)
packet_counts.to_csv("packet_count_per_flow.csv", index=False)

              flow_id  packet_count
0     281632906290924             2
1     281666626122137             2
2     281899248441052             1
3     281966559501047             2
4     281972289391021            22
...               ...           ...
5194  844147303995621             1
5195  844176614133988            53
5196  844211464564066             2
5197  844218974787320             2
5198  844340159445264             3

[5199 rows x 2 columns]
